# Milestone 3 — Frontier-baseline failure analysis

The frontier baseline scored 4/12 on the working-development set and 11/24 on
the sealed frozen set. This notebook answers the diagnostic question that must
come before small-model optimization:

> Why did the frontier baseline fail, and which layer most likely owns each failure?

This milestone is a **retrospective backfill**: initial SLM experimentation had
already started when the missing PRD step was noticed. To preserve the stronger
leakage policy established in Milestone 1, only the eight failed `working_dev`
examples are inspected here. Individual frozen questions, generated SQL, and
outcomes remain sealed.

No model request is made in this notebook.


## Separate symptoms, causes, and ownership

A useful taxonomy should not call every SQLite error an “execution failure” and
stop there. Each record uses three dimensions:

| Dimension | Meaning | Examples |
|---|---|---|
| Observed failure | What the evaluator saw | wrong rows, wrong columns, syntax error |
| Root cause | Why the SQL was wrong | stored-value mapping, column selection, calculation |
| Likely owner | Where an intervention belongs | context, model reasoning, mixed, benchmark/data |

Categories are evidence-backed but not metaphysical truth. A primary cause is
selected for counting, while contributing causes preserve nuance.


In [1]:
from collections import Counter
from hashlib import sha256
from pathlib import Path
import json
import sqlite3

from domain_adapt.evaluation import compare_sql_results
from domain_adapt.finch import make_reference_compatible


DATA_DIR = Path("../data").resolve()
DATASET_PATH = DATA_DIR / "finch_dataset.json"
DB_PATH = DATA_DIR / "financial.sqlite"
MANIFEST_PATH = Path("../configs/eval_manifest_v1.json").resolve()
FRONTIER_RUN_PATH = DATA_DIR / "runs" / "gpt_5_2_minimal_working_dev_v1.json"

required_paths = (DATASET_PATH, DB_PATH, MANIFEST_PATH, FRONTIER_RUN_PATH)
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing failure-analysis prerequisites: {missing_paths}")

records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
frontier_run = json.loads(FRONTIER_RUN_PATH.read_text(encoding="utf-8"))
conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)
assert frontier_run["split"] == "working_dev"

record_by_id = {
    row["question_id"]: row
    for row in records
    if row["db_name"] == "bird" and row["db_id"] == "financial"
}
working_results = frontier_run["results"]
assert {result["question_id"] for result in working_results} == working_ids
assert not ({result["question_id"] for result in working_results} & frozen_ids)

failures = [result for result in working_results if not result["passed"]]
assert len(failures) == 8

print("Frontier run SHA-256:", sha256(FRONTIER_RUN_PATH.read_bytes()).hexdigest())
print("Working results:", len(working_results))
print("Failures selected for analysis:", len(failures))
print("Frozen records inspected: 0")


Frontier run SHA-256: ce46a21675356f5d51712f1e6f7df31267521e760ae219ba6d0138dd6b802ad4
Working results: 12
Failures selected for analysis: 8
Frozen records inspected: 0


## Evidence view

For each failure, compare the question, reference-compatible SQL, generated SQL,
evaluator reason, and a small result preview. The compatibility transformation
is applied only to reference SQL, exactly as in the evaluator.


In [2]:
def query_preview(sql: str, limit: int = 5):
    try:
        cursor = conn.execute(sql)
        rows = cursor.fetchmany(limit)
        return {
            "columns": [item[0] for item in cursor.description or []],
            "rows": rows,
        }
    except Exception as error:
        return {"error": f"{type(error).__name__}: {error}"}


evidence_by_id = {}

for failure in failures:
    question_id = failure["question_id"]
    example = record_by_id[question_id]
    reference_sql, rules = make_reference_compatible(example["SQL"])
    comparison = compare_sql_results(
        conn=conn,
        reference_sql=reference_sql,
        candidate_sql=failure["generated_sql"],
    )
    assert comparison.equivalent is False
    assert comparison.reason == failure["reason"]

    evidence_by_id[question_id] = {
        "question": example["question"],
        "reference_sql": reference_sql,
        "generated_sql": failure["generated_sql"],
        "reference_rules": rules,
        "evaluator_reason": comparison.reason,
        "reference_preview": query_preview(reference_sql),
        "candidate_preview": query_preview(failure["generated_sql"]),
    }

    evidence = evidence_by_id[question_id]
    print("=" * 88)
    print("Question ID:", question_id)
    print("Question:", evidence["question"])
    print("Reference SQL:\n", evidence["reference_sql"])
    print("Generated SQL:\n", evidence["generated_sql"])
    print("Evaluator reason:", evidence["evaluator_reason"])
    print("Reference preview:", evidence["reference_preview"])
    print("Candidate preview:", evidence["candidate_preview"])


Question ID: 11
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Reference SQL:
 SELECT t1.account_id FROM loan AS t1 INNER JOIN account AS t2 ON t1.account_id = t2.account_id WHERE STRFTIME('%Y', t2.date) = '1993' AND t1.duration > 12 ORDER BY t1.amount DESC LIMIT 1
Generated SQL:
 SELECT a.account_id
FROM account AS a
JOIN loan AS l ON l.account_id = a.account_id
WHERE l.duration > 12
  AND l.status = 'A'
  AND strftime('%Y', a.date) = '1993'
  AND l.amount = (
    SELECT MAX(l2.amount)
    FROM loan AS l2
    JOIN account AS a2 ON a2.account_id = l2.account_id
    WHERE l2.duration > 12
      AND l2.status = 'A'
      AND strftime('%Y', a2.date) = '1993'
  );
Evaluator reason: ordered results differ
Reference preview: {'columns': ['account_id'], 'rows': [(10451,)]}
Candidate preview: {'columns': ['account_id'], 'rows': [(5120,)]}
Question ID: 14
Question: State 

## Manual evidence-backed labels

These labels describe the first frontier run; they are not inferred from the
later Ministral experiments. `context_addressable` means that clearer schema or
value context could plausibly prevent the error. It does not guarantee that a
smaller model will use that context correctly.


In [3]:
manual_labels = {
    11: {
        "primary_root_cause": "unrequested_assumption",
        "contributing_causes": ["domain_semantic_mapping"],
        "likely_owner": "model_reasoning",
        "confidence": "high",
        "evidence": (
            "The generated SQL added loan.status = 'A', although the question "
            "requires only duration > 12, a 1993 account opening, and the "
            "largest approved amount. The extra status filter changes the winner."
        ),
    },
    14: {
        "primary_root_cause": "column_selection",
        "contributing_causes": ["aggregation_logic"],
        "likely_owner": "context_addressable",
        "confidence": "high",
        "evidence": (
            "The reference filters trans.amount > 3000. The generated SQL instead "
            "tests MAX(trans.balance) > 3000, confusing transaction amount with "
            "post-transaction account balance."
        ),
    },
    45: {
        "primary_root_cause": "opaque_column_mapping",
        "contributing_causes": ["requested_output_shape"],
        "likely_owner": "context_addressable",
        "confidence": "high",
        "evidence": (
            "The generated SQL ranks A12, the 1995 unemployment field, rather "
            "than A13 for 1996. It also returns A2 and A3 instead of district_id "
            "and A2."
        ),
    },
    49: {
        "primary_root_cause": "domain_semantic_mapping",
        "contributing_causes": ["incomplete_predicate"],
        "likely_owner": "context_addressable",
        "confidence": "high",
        "evidence": (
            "The generated SQL treats 'Branch location 1' as district.A2 text "
            "instead of district_id = 1, and treats running as only status C "
            "instead of both C and D."
        ),
    },
    61: {
        "primary_root_cause": "domain_semantic_mapping",
        "contributing_causes": ["schema_linking", "benchmark_wording_ambiguity"],
        "likely_owner": "context_addressable",
        "confidence": "medium",
        "evidence": (
            "The benchmark maps loan eligibility to disposition ownership: "
            "non-OWNER disp.type values are requested. The generated SQL instead "
            "returns account.frequency and defines ineligibility as having no loan."
        ),
    },
    81: {
        "primary_root_cause": "numeric_expression",
        "contributing_causes": ["output_unit"],
        "likely_owner": "model_reasoning",
        "confidence": "high",
        "evidence": (
            "The join and yearly totals are sensible, but the generated formula "
            "returns a ratio. The reference defines the growth rate as a percentage "
            "and multiplies by 100."
        ),
    },
    99: {
        "primary_root_cause": "stored_value_mapping",
        "contributing_causes": [],
        "likely_owner": "context_addressable",
        "confidence": "high",
        "evidence": (
            "The generated SQL uses the English phrase 'weekly issuance'. The "
            "database stores the exact frequency code 'POPLATEK TYDNE'."
        ),
    },
    105: {
        "primary_root_cause": "requested_output_shape",
        "contributing_causes": ["schema_linking", "unrequested_assumption"],
        "likely_owner": "mixed",
        "confidence": "high",
        "evidence": (
            "The benchmark asks for client ID, account district ID, and district "
            "name. The generated SQL returns only client ID and the client's home "
            "district, then adds an unnecessary NOT IN restriction."
        ),
    },
}


In [4]:
failure_ids = {failure["question_id"] for failure in failures}
assert set(manual_labels) == failure_ids

analysis_records = []
for failure in failures:
    question_id = failure["question_id"]
    label = manual_labels[question_id]
    assert label["primary_root_cause"]
    assert label["likely_owner"]
    assert label["confidence"] in {"low", "medium", "high"}
    assert len(label["evidence"].split()) >= 12

    analysis_records.append({
        "question_id": question_id,
        "difficulty": failure["difficulty"],
        "observed_failure": failure["reason"],
        **label,
    })

print("Every failure has an evidence-backed category:", len(analysis_records))


Every failure has an evidence-backed category: 8


## Quantify the taxonomy

Because this is a deliberately small learning slice, counts are more honest than
broad population claims. The distribution identifies intervention hypotheses; it
does not prove that a particular prompt change will work.


In [5]:
root_cause_counts = Counter(
    record["primary_root_cause"] for record in analysis_records
)
owner_counts = Counter(
    record["likely_owner"] for record in analysis_records
)
confidence_counts = Counter(
    record["confidence"] for record in analysis_records
)

print("Primary root causes:", dict(root_cause_counts))
print("Likely owner:", dict(owner_counts))
print("Confidence:", dict(confidence_counts))

assert owner_counts == {
    "context_addressable": 5,
    "model_reasoning": 2,
    "mixed": 1,
}


Primary root causes: {'unrequested_assumption': 1, 'column_selection': 1, 'opaque_column_mapping': 1, 'domain_semantic_mapping': 2, 'numeric_expression': 1, 'stored_value_mapping': 1, 'requested_output_shape': 1}
Likely owner: {'model_reasoning': 2, 'context_addressable': 5, 'mixed': 1}
Confidence: {'high': 7, 'medium': 1}


In [6]:
print(f"{'ID':>3}  {'Observed failure':<31} {'Primary cause':<25} {'Owner'}")
print("-" * 100)
for record in sorted(analysis_records, key=lambda item: item["question_id"]):
    print(
        f"{record['question_id']:>3}  "
        f"{record['observed_failure']:<31} "
        f"{record['primary_root_cause']:<25} "
        f"{record['likely_owner']}"
    )


 ID  Observed failure                Primary cause             Owner
----------------------------------------------------------------------------------------------------
 11  ordered results differ          unrequested_assumption    model_reasoning
 14  unordered results differ        column_selection          context_addressable
 45  ordered results differ          opaque_column_mapping     context_addressable
 49  unordered results differ        domain_semantic_mapping   context_addressable
 61  unordered results differ        domain_semantic_mapping   context_addressable
 81  unordered results differ        numeric_expression        model_reasoning
 99  unordered results differ        stored_value_mapping      context_addressable
105  column count differs: 3 != 2    requested_output_shape    mixed


## Findings and hypotheses

Five of eight working failures are plausibly context-addressable:

- opaque fields such as `A12` versus `A13` need meanings;
- stored codes such as `POPLATEK TYDNE` need verified mappings;
- business phrases such as “running contract” and “eligible for loans” need
  definitions grounded in actual values and relationships.

Two failures look primarily like model reasoning/construction problems:

- adding a filter that the question never requested;
- returning a ratio when the expected unit is percentage.

One failure is mixed: additional schema meaning helps identify the correct
district, but returning the exact requested columns and avoiding an invented
restriction are also construction responsibilities.

This taxonomy justifies testing relationships, column meanings, verified values,
compact context, and general SQL-construction checks one at a time in Milestone 5.
It does **not** justify question-specific rules, few-shot examples selected from
these failures, or inspecting the sealed frozen set.


In [7]:
taxonomy_artifact = {
    "taxonomy_version": 1,
    "source_run": FRONTIER_RUN_PATH.name,
    "source_run_sha256": sha256(FRONTIER_RUN_PATH.read_bytes()).hexdigest(),
    "split": "working_dev",
    "sampled_failures": len(analysis_records),
    "frozen_records_inspected": 0,
    "root_cause_counts": dict(root_cause_counts),
    "owner_counts": dict(owner_counts),
    "records": sorted(analysis_records, key=lambda item: item["question_id"]),
}

taxonomy_path = DATA_DIR / "runs" / "frontier_failure_taxonomy_working_dev_v1.json"
taxonomy_path.parent.mkdir(parents=True, exist_ok=True)
temporary_path = taxonomy_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(taxonomy_artifact, indent=2), encoding="utf-8")
temporary_path.replace(taxonomy_path)

print("Saved:", taxonomy_path)
print("Taxonomy SHA-256:", sha256(taxonomy_path.read_bytes()).hexdigest())
print("Validation passed: every sampled failure has an evidence-backed category.")
print("No model request made.")


Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\frontier_failure_taxonomy_working_dev_v1.json
Taxonomy SHA-256: b0795fd6141ffb07560d25301502be21b52b92d8495daa2efc28e093301a190b
Validation passed: every sampled failure has an evidence-backed category.
No model request made.
